In [53]:
# Cell 1 — imports
import os
import pandas as pd
import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms


In [54]:
# Cell 2 - Enhanced Augmentation Classes
class RandomMorph(object):
    def __init__(self, p=0.8):
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.rand() < self.p:
            arr = np.array(img)
            kernel_size = np.random.choice([3, 5])
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            iterations = np.random.randint(1, 3)

            if np.random.rand() < 0.5:
                arr = cv2.dilate(arr, kernel, iterations=iterations)
            else:
                arr = cv2.erode(arr, kernel, iterations=iterations)
            return Image.fromarray(arr)
        return img

In [55]:
# Cell 3 - Enhanced Transforms (CORRECTED)
child_transform = transforms.Compose([
    # Much more aggressive geometric transformations
    transforms.RandomAffine(
        degrees=20,  # ±20 degrees rotation
        translate=(0.2, 0.2),  # ±20% translation
        scale=(0.8, 1.3),  # 80%-130% scaling
        shear=15,  # ±15 degrees shear
        fill=255
    ),

    # Additional perspective distortion
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),

    # Morphological operations
    RandomMorph(p=0.8),

    # Convert to tensor
    transforms.ToTensor(),

    # Intensity variations - FIXED: Use RandomApply instead of p parameter
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.4, contrast=0.4)
    ], p=0.7),

    # Add gaussian noise
    transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.08 if np.random.rand() < 0.5 else x),

    # Random erasing (simulate occlusion)
    transforms.RandomErasing(p=0.4, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=1.0),

    # Ensure values stay in [0,1] range
    transforms.Lambda(lambda x: torch.clamp(x, 0, 1))
])

original_transform = transforms.Compose([transforms.ToTensor()])

print("Enhanced transforms created successfully!")
print("Key augmentations:")
print("- Rotation: ±20°")
print("- Translation: ±20%")
print("- Scaling: 80%-130%")
print("- Shear: ±15°")
print("- Perspective distortion: 30%")
print("- Morphological operations: 80% chance")
print("- Color jitter: 70% chance")
print("- Gaussian noise: 50% chance")
print("- Random erasing: 40% chance")

Enhanced transforms created successfully!
Key augmentations:
- Rotation: ±20°
- Translation: ±20%
- Scaling: 80%-130%
- Shear: ±15°
- Perspective distortion: 30%
- Morphological operations: 80% chance
- Color jitter: 70% chance
- Gaussian noise: 50% chance
- Random erasing: 40% chance


In [56]:
# Cell 4 — dataset class
class ShapeScoreDataset(Dataset):
    def __init__(self, csv_path, images_dir, shape_id,
                 child_transform, original_transform):
        # load & filter
        df = pd.read_csv(csv_path)
        df = df[df['shape'] == shape_id]

        # build list of (child_png_path, score)
        self.records = []
        for _, row in df.iterrows():
            cid, score = int(row['child_id']), int(row['score']) - 1  # to 0/1/2
            fn = os.path.join(images_dir, f"{cid}.png")
            if os.path.exists(fn):
                self.records.append((fn, score))

        # load the fixed original image once
        orig_fn = os.path.join(images_dir, f"original{shape_id}.png")
        self.orig_img = Image.open(orig_fn).convert('L')
        self.orig_img = original_transform(self.orig_img)

        self.child_transform = child_transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        img_path, score = self.records[idx]
        child = Image.open(img_path).convert('L')
        child = self.child_transform(child)
        # stack child + original → 2×H×W
        x = torch.cat([child, self.orig_img], dim=0)
        return x, score


In [57]:
# Cell 5 - Create Enhanced Dataset (3x effective size)
class MultiAugmentDataset(Dataset):
    def __init__(self, base_dataset, num_augmentations=4):  # 4x the data
        self.base_dataset = base_dataset
        self.num_augmentations = num_augmentations

    def __len__(self):
        return len(self.base_dataset) * self.num_augmentations

    def __getitem__(self, idx):
        base_idx = idx // self.num_augmentations
        return self.base_dataset[base_idx]

# Create original dataset
dataset = ShapeScoreDataset(csv_path, images_dir, shape_id=2,
                           child_transform=child_transform,
                           original_transform=original_transform)

# Wrap with multi-augmentation to effectively 4x your data
enhanced_dataset = MultiAugmentDataset(dataset, num_augmentations=4)

# Split
train_size = int(0.8 * len(enhanced_dataset))
val_size = len(enhanced_dataset) - train_size
train_ds, val_ds = random_split(enhanced_dataset, [train_size, val_size])

# Smaller batches for more frequent updates
train_loader = DataLoader(train_ds, batch_size=6, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=6, shuffle=False, num_workers=0)

print(f"Original dataset: {len(dataset)} samples")
print(f"Enhanced dataset: {len(enhanced_dataset)} samples (4x augmentation)")
print(f"Train: {train_size}, Val: {val_size}")

Original dataset: 286 samples
Enhanced dataset: 1144 samples (4x augmentation)
Train: 915, Val: 229


In [58]:
# Cell 6 — define a small 2-channel CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # after 3×pool2: 200→100→50→25
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*25*25, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [59]:
# Cell 7 - Enhanced Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()

# Better optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.7, patience=4)

# Training with early stopping
num_epochs = 30
best_val_loss = float('inf')
patience = 8
patience_counter = 0

for epoch in range(1, num_epochs + 1):
    # Training
    model.train()
    train_loss = 0.0
    train_samples = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        train_samples += xb.size(0)

    train_loss /= train_samples

    # Validation
    model.eval()
    val_loss = 0.0
    correct = total = 0
    val_samples = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * xb.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
            val_samples += xb.size(0)

    val_loss /= val_samples
    val_acc = correct / total

    # Learning rate scheduling
    scheduler.step(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_shape_model.pth')
    else:
        patience_counter += 1

    print(f"Epoch {epoch:2d}/{num_epochs} — "
          f"Train: {train_loss:.4f}  "
          f"Val: {val_loss:.4f}  "
          f"Acc: {val_acc:.4f}  "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# Load best model
model.load_state_dict(torch.load('best_shape_model.pth'))
print("Training completed with enhanced augmentation!")

Epoch  1/30 — Train: 1.0061  Val: 0.9553  Acc: 0.5983  LR: 0.000100
Epoch  2/30 — Train: 1.0010  Val: 0.9536  Acc: 0.5983  LR: 0.000100
Epoch  3/30 — Train: 1.0043  Val: 0.9393  Acc: 0.5983  LR: 0.000100
Epoch  4/30 — Train: 1.0094  Val: 0.9574  Acc: 0.5983  LR: 0.000100
Epoch  5/30 — Train: 1.0107  Val: 0.9560  Acc: 0.5983  LR: 0.000100
Epoch  6/30 — Train: 1.0050  Val: 0.9666  Acc: 0.5983  LR: 0.000100
Epoch  7/30 — Train: 1.0044  Val: 0.9460  Acc: 0.5983  LR: 0.000100
Epoch  8/30 — Train: 1.0017  Val: 0.9483  Acc: 0.5983  LR: 0.000070
Epoch  9/30 — Train: 0.9972  Val: 0.9567  Acc: 0.5983  LR: 0.000070
Epoch 10/30 — Train: 0.9987  Val: 0.9536  Acc: 0.5983  LR: 0.000070
Epoch 11/30 — Train: 1.0048  Val: 0.9470  Acc: 0.5983  LR: 0.000070
Early stopping at epoch 11
Training completed with enhanced augmentation!


In [60]:
# First, let's diagnose the issues
def diagnose_dataset():
    """Check dataset balance and sample a few examples"""
    print("=== DATASET DIAGNOSIS ===")

    # Check original dataset size and distribution
    orig_dataset = ShapeScoreDataset(csv_path, images_dir, shape_id=2,
                                   child_transform=original_transform,  # No augmentation for diagnosis
                                   original_transform=original_transform)

    print(f"Original dataset size: {len(orig_dataset)}")

    # Check class distribution
    scores = []
    for i in range(len(orig_dataset)):
        _, score = orig_dataset[i]
        scores.append(score)

    unique, counts = np.unique(scores, return_counts=True)
    print("Class distribution:")
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count} samples ({count/len(scores)*100:.1f}%)")

    # Check if classes are severely imbalanced
    if len(unique) < 3:
        print("⚠️  WARNING: Not all 3 classes present in dataset!")

    if max(counts) / min(counts) > 5:
        print("⚠️  WARNING: Severe class imbalance detected!")

    return orig_dataset, scores

# Run diagnosis
orig_dataset, scores = diagnose_dataset()

# FIXED DATASET CLASS with better error handling
class ImprovedShapeScoreDataset(Dataset):
    def __init__(self, csv_path, images_dir, shape_id, child_transform, original_transform):
        # Load and filter data
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"CSV file not found: {csv_path}")

        df = pd.read_csv(csv_path)
        print(f"Total CSV records: {len(df)}")

        df = df[df['shape'] == shape_id]
        print(f"Records for shape {shape_id}: {len(df)}")

        if len(df) == 0:
            raise ValueError(f"No records found for shape {shape_id}")

        # Build records with validation
        self.records = []
        missing_files = []

        for _, row in df.iterrows():
            try:
                cid = int(row['child_id'])
                score = int(row['score']) - 1  # Convert to 0/1/2

                # Validate score range
                if score not in [0, 1, 2]:
                    print(f"Warning: Invalid score {score+1} for child {cid}, skipping...")
                    continue

                fn = os.path.join(images_dir, f"{cid}.png")
                if os.path.exists(fn):
                    self.records.append((fn, score))
                else:
                    missing_files.append(fn)

            except (ValueError, KeyError) as e:
                print(f"Error processing row {row}: {e}")
                continue

        print(f"Valid records found: {len(self.records)}")
        print(f"Missing files: {len(missing_files)}")

        if len(self.records) == 0:
            raise ValueError("No valid records found!")

        # Load original image
        orig_fn = os.path.join(images_dir, f"original{shape_id}.png")
        if not os.path.exists(orig_fn):
            raise FileNotFoundError(f"Original image not found: {orig_fn}")

        try:
            self.orig_img = Image.open(orig_fn).convert('L')
            print(f"Original image size: {self.orig_img.size}")
            self.orig_img = original_transform(self.orig_img)
        except Exception as e:
            raise ValueError(f"Error loading original image {orig_fn}: {e}")

        self.child_transform = child_transform

        # Print class distribution
        class_counts = {}
        for _, score in self.records:
            class_counts[score] = class_counts.get(score, 0) + 1

        print("Class distribution in dataset:")
        for cls in sorted(class_counts.keys()):
            count = class_counts[cls]
            pct = count / len(self.records) * 100
            print(f"  Class {cls}: {count} samples ({pct:.1f}%)")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        try:
            img_path, score = self.records[idx]
            child = Image.open(img_path).convert('L')
            child = self.child_transform(child)
            x = torch.cat([child, self.orig_img], dim=0)
            return x, score
        except Exception as e:
            print(f"Error loading sample {idx}: {e}")
            # Return dummy sample to prevent crash
            dummy = torch.zeros_like(self.orig_img)
            x = torch.cat([dummy, self.orig_img], dim=0)
            return x, 0

print("\n=== CREATING IMPROVED DATASET ===")

# Create improved dataset WITHOUT multi-augmentation first
improved_dataset = ImprovedShapeScoreDataset(
    csv_path, images_dir, shape_id=2,
    child_transform=child_transform,
    original_transform=original_transform
)

print(f"\nImproved dataset created with {len(improved_dataset)} samples")

# Split the dataset
train_size = int(0.8 * len(improved_dataset))
val_size = len(improved_dataset) - train_size
train_ds, val_ds = random_split(improved_dataset, [train_size, val_size])

print(f"Train size: {train_size}, Val size: {val_size}")

# Create data loaders with better settings
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, drop_last=False)

print("Data loaders created successfully!")

=== DATASET DIAGNOSIS ===
Original dataset size: 286
Class distribution:
  Class 0: 70 samples (24.5%)
  Class 1: 162 samples (56.6%)
  Class 2: 54 samples (18.9%)

=== CREATING IMPROVED DATASET ===
Total CSV records: 1346
Records for shape 2: 289
Valid records found: 286
Missing files: 3
Original image size: (200, 200)
Class distribution in dataset:
  Class 0: 70 samples (24.5%)
  Class 1: 162 samples (56.6%)
  Class 2: 54 samples (18.9%)

Improved dataset created with 286 samples
Train size: 228, Val size: 58
Data loaders created successfully!


In [ ]:
# =============================================================================
# SHAPE SCORING NOTEBOOK — COMPLETE, IMPROVED VERSION
# Classification of children drawings into 3 quality bands (scores 1‑3)
# *Key improvements*
#   • Robust handling of class‑imbalance (weighted loss  + optional oversampling)
#   • More meaningful input representation: [child, reference, |child‑reference|]
#   • Two model options: a light CNN **or** transfer‑learning with ResNet‑18
#   • Cleaner augmentation pipeline – only geometry / morphology (no colour jitter)
#   • Modular design to tweak quickly in research iterations
# =============================================================================

# -------------------------------------------------
# Cell 1 — Imports & global config
# -------------------------------------------------
import os, warnings, math, random
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import models

import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"✅ Imports ready — PyTorch {torch.__version__}, CUDA={torch.cuda.is_available()}")

# -------------------------------------------------
# Cell 2 — Custom transforms
# -------------------------------------------------
class RandomMorph:
    """Apply random dilation / erosion to thicken or thin strokes"""
    def __init__(self, p: float = .5):
        self.p = p

    def __call__(self, img: Image.Image):
        if np.random.rand() > self.p:
            return img
        arr = np.array(img)
        k = np.random.choice([3, 5])
        kernel = np.ones((k, k), np.uint8)
        iters = np.random.randint(1, 2)
        if np.random.rand() < .5:
            arr = cv2.dilate(arr, kernel, iters)
        else:
            arr = cv2.erode(arr, kernel, iters)
        return Image.fromarray(arr)

class GaussianNoise:
    """Additive Gaussian noise on tensor image (0‑1 range)"""
    def __init__(self, std: float = .05, p: float = .3):
        self.std, self.p = std, p

    def __call__(self, t: torch.Tensor):
        if torch.rand(1) > self.p:
            return t
        noise = torch.randn_like(t) * self.std
        return torch.clamp(t + noise, 0., 1.)

# -------------------------------------------------
# Cell 3 — Augmentation pipelines
# -------------------------------------------------
child_tf = T.Compose([
    T.RandomAffine(degrees=15, translate=(.1, .1), scale=(.9, 1.1), shear=10, fill=0),
    T.RandomPerspective(distortion_scale=.2, p=.4),
    RandomMorph(p=.6),
    T.ToTensor(),
    GaussianNoise(std=.05, p=.3),
    # simulate partial occlusion
    T.RandomErasing(p=.25, scale=(.02, .07), ratio=(.5, 1.5), value=0),
])

orig_tf = T.ToTensor()  # just convert to 0‑1 tensor

print("✅ Transforms ready")

# -------------------------------------------------
# Cell 4 — Dataset
# -------------------------------------------------
class ShapeScoreDataset(Dataset):
    """Return tensor with 3 channels: child, reference, |child‑reference| & label"""
    def __init__(self, csv_path: str, images_dir: str, shape_id: int,
                 child_transform, orig_transform, verbose: bool = True):
        self.records = []
        df = pd.read_csv(csv_path)
        df = df[df['shape'] == shape_id]
        if verbose:
            print(f"🔍 Data rows for shape {shape_id}: {len(df)}")

        # load reference image once
        ref_path = os.path.join(images_dir, f"original{shape_id}.png")
        if not os.path.exists(ref_path):
            raise FileNotFoundError(ref_path)
        self.ref_img = orig_transform(Image.open(ref_path).convert('L'))  # 1×H×W
        self.child_tf, self.orig_tf = child_transform, orig_transform

        for _, row in df.iterrows():
            cid   = int(row['child_id'])
            score = int(row['score']) - 1  # map 1‑3 → 0‑2
            if score not in {0,1,2}:
                continue
            ipath = os.path.join(images_dir, f"{cid}.png")
            if os.path.exists(ipath):
                self.records.append((ipath, score))
        if verbose:
            cnts = Counter([s for _, s in self.records])
            print("📊 Class dist:", cnts)
        if not self.records:
            raise RuntimeError("No valid images found")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        path, y = self.records[idx]
        child = Image.open(path).convert('L')
        child = self.child_tf(child)
        ref   = self.ref_img  # already tensor
        diff  = torch.abs(child - ref)
        x = torch.cat([child, ref, diff], dim=0)  # 3×H×W
        return x, y

print("✅ Dataset class ready")

# -------------------------------------------------
# Cell 5 — Utility: class weights & sampler
# -------------------------------------------------

def make_loaders(dataset: Dataset, batch_size: int = 16, oversample: bool = True):
    train_len = int(.8 * len(dataset))
    val_len   = len(dataset) - train_len
    train_ds, val_ds = random_split(dataset, [train_len, val_len],
                                    generator=torch.Generator().manual_seed(SEED))

    # --- class weights (for loss) computed on *train* split only
    train_labels = [dataset[i][1] for i in train_ds.indices]
    label_counts = Counter(train_labels)
    num_samples  = sum(label_counts.values())
    class_wts = {cls: num_samples/count for cls, count in label_counts.items()}
    weight_tensor = torch.tensor([class_wts[c] for c in sorted(class_wts)])

    # --- optional oversampling so each mini‑batch is balanced
    if oversample:
        sample_wts = [class_wts[train_labels[i]] for i in range(len(train_labels))]
        sampler = WeightedRandomSampler(sample_wts, num_samples=len(train_labels), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, weight_tensor

# -------------------------------------------------
# Cell 6 — Models
# -------------------------------------------------
class SmallCNN(nn.Module):
    def __init__(self, n_classes=3, in_ch=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(.3),
            nn.Linear(256, n_classes)
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def get_resnet18(n_classes=3):
    """Pre‑trained ResNet‑18 adapted for 3‑channel 200×200 monochrome drawings"""
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # first conv already 3‑ch, we just upscale our 200×200 to 224 and feed
    m.conv1.stride = (1,1)  # keep more spatial info (optional)
    m.avgpool = nn.AdaptiveAvgPool2d((1,1))
    m.fc = nn.Linear(m.fc.in_features, n_classes)
    return m

print("✅ Model definitions ready")

# -------------------------------------------------
# Cell 7 — Training utilities
# -------------------------------------------------

def accuracy(outputs, labels):
    _, preds = outputs.max(1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optim, loss_fn, device):
    model.train()
    running = 0.
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optim.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optim.step()
        running += loss.item() * x.size(0)
    return running / len(loader.dataset)

def evaluate(model, loader, loss_fn, device):
    model.eval();  loss_total, acc_total = 0., 0.
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            loss_total += loss_fn(out, y).item() * x.size(0)
            acc_total  += (out.argmax(1)==y).sum().item()
    return loss_total/len(loader.dataset), acc_total/len(loader.dataset)

# -------------------------------------------------
# Cell 8 — Main training loop
# -------------------------------------------------

def fit(model, train_loader, val_loader, loss_fn, epochs=25, lr=2e-4, patience=7):
    device = next(model.parameters()).device
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    best_val, counter = math.inf, 0
    history = {'tr_loss':[], 'vl_loss':[], 'vl_acc':[]}

    for epoch in range(1, epochs+1):
        tr = train_one_epoch(model, train_loader, opt, loss_fn, device)
        vl, acc = evaluate(model, val_loader, loss_fn, device)
        sched.step()
        history['tr_loss'].append(tr); history['vl_loss'].append(vl); history['vl_acc'].append(acc)
        print(f"E{epoch:02d}: train {tr:.3f}  |  val {vl:.3f}  |  acc {acc:.3f}")
        # early stop
        if vl < best_val - 1e-4:
            best_val, counter = vl, 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            counter += 1
            if counter >= patience:
                print("⏹️  Early stopping")
                break
    return history

# -------------------------------------------------
# Cell 9 — RUN!  (edit paths below)
# -------------------------------------------------
CSV_PATH   = r"C:\Users\yozev\PycharmProjects\finetuning\child_shape_scores.csv"
IMG_DIR    = r"C:\Users\yozev\OneDrive\Desktop\Shapes2\shape2"
SHAPE_ID   = 2
BATCH_SIZE = 16

full_ds = ShapeScoreDataset(CSV_PATH, IMG_DIR, SHAPE_ID, child_tf, orig_tf)
train_loader, val_loader, cls_weights = make_loaders(full_ds, batch_size=BATCH_SIZE, oversample=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_choice = 'resnet'   # 'small' or 'resnet'

if model_choice == 'small':
    model = SmallCNN(in_ch=3).to(DEVICE)
else:
    model = get_resnet18().to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=cls_weights.to(DEVICE))
print("🏁 Starting training…\n")

hist = fit(model, train_loader, val_loader, criterion, epochs=30, lr=2e-4, patience=10)

# -------------------------------------------------
# Cell 10 — Plot history
# -------------------------------------------------
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.plot(hist['tr_loss'], label='train'); plt.plot(hist['vl_loss'], label='val'); plt.legend(); plt.title('Loss')
plt.subplot(1,3,2); plt.plot(hist['vl_acc']); plt.title('Val accuracy'); plt.ylim(0,1)
plt.subplot(1,3,3); plt.plot([p['lr'] for p in optim.AdamW(model.parameters()).param_groups]); plt.title('LR (initial)');
plt.tight_layout(); plt.show()

# -------------------------------------------------
# Cell 11 — Final evaluation + per‑class stats
# -------------------------------------------------
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
model.eval();
all_preds, all_labels = [], []
with torch.no_grad():
    for x,y in val_loader:
        out = model(x.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

cm = Counter([(l,p) for l,p in zip(all_labels, all_preds)])
print("\nConfusion counts (label→pred):", cm)

acc_overall = sum(1 for l,p in zip(all_labels, all_preds) if l==p) / len(all_labels)
print(f"Overall accuracy: {acc_overall:.3f}  ({len(all_labels)} samples)")

for c in sorted(set(all_labels)):
    tp = cm.get((c,c),0)
    tot = sum(cm.get((c,p),0) for p in range(3))
    print(f"Class {c}: {tp}/{tot}  accuracy {tp/tot if tot else 0:.3f}")

print("\n🎉 Done!  Best model saved to best_model.pth")


C:\Users\yozev\AppData\Roaming\Python\Python311\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


✅ Imports ready — PyTorch 2.7.0+cpu, CUDA=False
✅ Transforms ready
✅ Dataset class ready
✅ Model definitions ready
🔍 Data rows for shape 2: 289
📊 Class dist: Counter({1: 162, 0: 70, 2: 54})
🏁 Starting training…

E01: train 1.052  |  val 1.609  |  acc 0.207
E02: train 0.815  |  val 1.568  |  acc 0.345
E03: train 0.771  |  val 0.877  |  acc 0.414
E04: train 0.898  |  val 1.055  |  acc 0.414
E05: train 0.858  |  val 1.162  |  acc 0.259
E06: train 0.749  |  val 0.997  |  acc 0.345
E07: train 0.935  |  val 1.065  |  acc 0.328
E08: train 0.666  |  val 0.996  |  acc 0.362
E09: train 0.736  |  val 1.054  |  acc 0.448
E10: train 0.741  |  val 1.197  |  acc 0.293
E11: train 0.753  |  val 1.136  |  acc 0.293
E12: train 0.667  |  val 0.983  |  acc 0.397
E13: train 0.725  |  val 0.994  |  acc 0.397
⏹️  Early stopping


In [5]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
